# The specification for LAUs was stopped in 2021, so we have to manually combine the LAUs of Europe and the LADs (Local administrative Districts) of great britain and those of northern Ireland into a combined dataset

In [ ]:
import geopandas as gpd
import os
import os.path as path
import utils

In [ ]:
uk_shapes = gpd.read_file(utils.uk_lau_dir)
eu_shapes = gpd.read_file(utils.eu_lau_dir)
ie_shapes = gpd.read_file(utils.ie_lau_dir)

# EU data was downloaded in 4326
uk_shapes = uk_shapes.to_crs(epsg=4326) 
ie_shapes = ie_shapes.to_crs(epsg=4326)
assert eu_shapes.crs == uk_shapes.crs
assert eu_shapes.crs == ie_shapes.crs

In [ ]:
# unify column names and add a country code
uk_shapes = uk_shapes.rename(columns={
    "LAD21CD": "LAU_ID",
    "LAD21NM": "LAU_NAME"
})
uk_shapes["CNTR_CODE"] = "UK"

In [ ]:
ie_shapes = ie_shapes.rename(columns={
    "SDZ2021_cd": "LAU_ID",
    "SDZ2021_nm": "LAU_NAME"
})
ie_shapes["CNTR_CODE"] = "UK"

In [ ]:
# combine the geodataframes into a single Frame and drop unused columns
europe_shapes = gpd.pd.concat([eu_shapes, uk_shapes, ie_shapes], ignore_index=True)
europe_shapes = europe_shapes[['CNTR_CODE', 'LAU_ID', 'LAU_NAME', 'geometry']]

In [ ]:
# write to a combined shapefile
if not path.isdir(utils.lau_dir):
    os.mkdir(utils.lau_dir)
europe_shapes.to_file(path.join(utils.lau_dir, path.basename(utils.lau_dir)), driver="ESRI Shapefile")